# 07. Експерименти з LSTM

**Етап 07 з 13** · NASA C-MAPSS FD001 · RUL prediction


## 1. Дані

In [ ]:
!pip install mlflow

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import mlflow

mlflow.set_tracking_uri(
    "https://stage-broom-alkalize.ngrok-free.dev"
)

experiment = mlflow.set_experiment(
    "Predictive Maintenance - CNN and LSTM"
)

print("Experiment ID:", experiment.experiment_id)
print("Tracking URI:", mlflow.get_tracking_uri())

In [5]:
import copy
import os
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Шлях до даних. На Colab: "/content/drive/MyDrive/predictive_maintenance"
DATA_DIR = Path(os.environ.get("FD001_DIR", "/content/drive/MyDrive/predictive_maintenance"))

SEED = 42
BATCH_SIZE = 64
HIDDEN_SIZE = 64
LR = 1e-3
# QUICK=1 — короткий тестовий прогін (до 2 епох на конфігурацію), лише щоб перевірити код
QUICK = os.environ.get("FD001_QUICK", "0") == "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE, "| QUICK:", QUICK)

Device: cuda | QUICK: False


In [6]:
COLUMNS = ["unit_id", "cycle", "setting_1", "setting_2", "setting_3"] + [f"sensor_{i}" for i in range(1, 22)]

# Базові ознаки: цикл, 3 налаштування і 21 сенсор (25 штук)
BASE_FEATURES = ["cycle", "setting_1", "setting_2", "setting_3"] + [f"sensor_{i}" for i in range(1, 22)]


def load_fd001(data_dir):
    """Читає train FD001 і додає RUL як кількість циклів до останнього циклу двигуна."""
    train = pd.read_csv(data_dir / "train_FD001.txt", sep=r"\s+", header=None, names=COLUMNS)
    train["RUL"] = train.groupby("unit_id")["cycle"].transform("max") - train["cycle"]
    return train


def split_engines(df):
    """Ділить рядки на train і validation (80/20) за двигунами, без перетину двигунів."""
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(splitter.split(df, groups=df["unit_id"]))
    return df.iloc[train_idx].copy(), df.iloc[val_idx].copy()


fd_train = load_fd001(DATA_DIR)
base_train, base_val = split_engines(fd_train)
print("Base: train", len(base_train), "рядків,", base_train["unit_id"].nunique(), "двигунів |",
      "validation", len(base_val), "рядків,", base_val["unit_id"].nunique(), "двигунів")

Base: train 16561 рядків, 80 двигунів | validation 4070 рядків, 20 двигунів


## 2. Ознаки

In [ ]:
train_new = pd.read_csv(DATA_DIR / "new_train_49_features.csv")
# Якщо cycle.1 дублює cycle, прибираємо дубль
if (train_new["cycle"] == train_new["cycle.1"]).all():
    train_new = train_new.drop(columns=["cycle.1"])

DEGRADATION_FEATURES = BASE_FEATURES + ["degradation_score"]

new_train, new_val = split_engines(train_new)
assert not set(new_train["unit_id"]) & set(new_val["unit_id"]), "двигуни перетинаються"
print("Degradation: train", new_train["unit_id"].nunique(), "двигунів |",
      "validation", new_val["unit_id"].nunique(), "двигунів")

Degradation: train 64 двигунів | validation 16 двигунів


In [8]:
def prepare_feature_pair(train_df, val_df, features, add_missing_indicators=True):
    """Заповнює пропуски нулями і додає індикатори пропусків, які визначені за train."""
    train = train_df.copy()
    val = val_df.copy()

    missing_features = [col for col in features if train[col].isna().any()]

    indicator_features = []
    if add_missing_indicators:
        for col in missing_features:
            indicator = f"{col}_was_missing"
            train[indicator] = train[col].isna().astype("int8")
            val[indicator] = val[col].isna().astype("int8")
            indicator_features.append(indicator)

    train[features] = train[features].fillna(0)
    val[features] = val[features].fillna(0)

    final_features = features + indicator_features
    assert train[final_features].isna().sum().sum() == 0
    assert val[final_features].isna().sum().sum() == 0
    assert np.isfinite(train[final_features].to_numpy()).all()
    assert np.isfinite(val[final_features].to_numpy()).all()

    return train, val, final_features


def build_dataset(train_raw, val_raw, features):
    """Готує набір ознак: пропуски, індикатори і StandardScaler, що фітиться тільки на train."""
    train, val, final_features = prepare_feature_pair(train_raw, val_raw, features, add_missing_indicators=True)

    scaler = StandardScaler()
    train[final_features] = scaler.fit_transform(train[final_features])
    val[final_features] = scaler.transform(val[final_features])

    return {"train": train, "val": val, "features": final_features, "scaler": scaler}


# Два набори ознак; кожен має свої train/validation з розбиттям за двигунами
DATASETS = {
    "base": build_dataset(base_train, base_val, BASE_FEATURES),
    "degradation": build_dataset(new_train, new_val, DEGRADATION_FEATURES),
}
for name, data in DATASETS.items():
    print(f"{name}: {len(data['features'])} ознак -> {data['features'][-2:]}")

base: 25 ознак -> ['sensor_20', 'sensor_21']
degradation: 27 ознак -> ['degradation_score', 'degradation_score_was_missing']


## 3. Експерименти

In [ ]:
def create_windows(df, feature_cols, window_size):
    """Збирає ковзні вікна по двигунах; ціль — RUL в останньому циклі вікна."""
    X, y = [], []
    for _, engine in df.groupby("unit_id", sort=False):
        engine = engine.sort_values("cycle")
        features = engine[feature_cols].to_numpy(dtype=np.float32)
        rul = engine["RUL"].to_numpy(dtype=np.float32)
        for end in range(window_size, len(engine) + 1):
            X.append(features[end - window_size:end])
            y.append(rul[end - 1])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


class LSTMRegressor(nn.Module):
    """Одношаровий LSTM; RUL передбачається з останнього прихованого стану."""

    def __init__(self, input_size, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])


def set_seed(seed=SEED):
    """Фіксує seed для random, numpy і torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def predict(model, loader):
    """Повертає (істина, передбачення) для всього loader у режимі eval."""
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            pred = model(X_batch.to(DEVICE))
            preds.extend(pred.cpu().numpy().ravel())
            actuals.extend(y_batch.numpy().ravel())
    return np.asarray(actuals), np.asarray(preds)


def metrics(actuals, preds):
    """Рахує MAE, MSE, RMSE і MAE лише для двигунів з RUL >= 150."""
    mse = mean_squared_error(actuals, preds)
    mask = actuals >= 150
    return {
        "mae": mean_absolute_error(actuals, preds),
        "mse": mse,
        "rmse": np.sqrt(mse),
        "high_rul_mae": mean_absolute_error(actuals[mask], preds[mask]) if mask.any() else np.nan,
    }


def run_experiment(cfg):
    """Навчає LSTM за конфігурацією, логує метрики в MLflow без завантаження важких артефактів."""
    data = DATASETS[cfg["features"]]
    X_tr, y_tr = create_windows(data["train"], data["features"], cfg["window"])
    X_va, y_va = create_windows(data["val"], data["features"], cfg["window"])

    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr).unsqueeze(1))
    val_ds = TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va).unsqueeze(1))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    set_seed(SEED)
    model = LSTMRegressor(len(data["features"]), HIDDEN_SIZE).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    epochs = min(cfg["epochs"], 2) if QUICK else cfg["epochs"]

    history = {k: [] for k in ["train_loss", "val_mse", "val_mae", "val_high_rul_mae"]}
    best = {"mae": float("inf"), "epoch": 0, "state": None}
    wait = 0

    with mlflow.start_run(run_name=cfg["name"]) as run:
        run_id = run.info.run_id

        # лог
        mlflow.log_params(cfg)
        mlflow.log_params({
            "hidden_size": HIDDEN_SIZE,
            "learning_rate": LR,
            "batch_size": BATCH_SIZE,
            "seed": SEED,
            "num_features": len(data["features"])
        })

        for epoch in range(1, epochs + 1):
            model.train()
            running_loss = 0.0
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(X_batch.to(DEVICE)), y_batch.to(DEVICE))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                running_loss += loss.item() * X_batch.size(0)

            epoch_train_loss = running_loss / len(train_ds)
            va = metrics(*predict(model, val_loader))

            history["train_loss"].append(epoch_train_loss)
            history["val_mse"].append(va["mse"])
            history["val_mae"].append(va["mae"])
            history["val_high_rul_mae"].append(va["high_rul_mae"])

            # лог метрик
            mlflow.log_metrics({
                "train_loss": epoch_train_loss,
                "val_mse": va["mse"],
                "val_mae": va["mae"],
                "val_high_rul_mae": va["high_rul_mae"]
            }, step=epoch)

            print(f"{cfg['name']} | ep {epoch:02d} | Train Loss {epoch_train_loss:.3f} | Val MAE {va['mae']:.3f}")

            if va["mae"] < best["mae"]:
                best.update(mae=va["mae"], epoch=epoch, state=copy.deepcopy(model.state_dict()))
                wait = 0
            else:
                wait += 1
            if wait >= cfg["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

        # Відновлення кращих ваг для підсумку
        model.load_state_dict(best["state"])
        actuals, preds = predict(model, val_loader)
        final = metrics(actuals, preds)
        final_r2 = r2_score(actuals, preds)

        row = {
            "name": cfg["name"],
            "features": cfg["features"],
            "window": cfg["window"],
            "max_epochs": cfg["epochs"],
            "patience": cfg["patience"],
            "best_epoch": best["epoch"],
            "val_mae": final["mae"],
            "val_rmse": final["rmse"],
            "val_r2": final_r2,
            "high_rul_mae_150plus": final["high_rul_mae"],
            "run_id": run_id
        }

        # підсумкові мтерики
        mlflow.log_metrics({
            "best_epoch": best["epoch"],
            "best_val_mae": final["mae"],
            "best_val_rmse": final["rmse"],
            "best_val_r2": final_r2,
            "best_high_rul_mae_150plus": final["high_rul_mae"]
        })

    return {
        "cfg": cfg,
        "row": row,
        "history": history,
        "state": best["state"],
        "model": model,
        "actual": actuals,
        "pred": preds
    }


def run_grid_and_save_top_k(configs, top_k=1):
    """Запускає grid search і зберігає MLflow артефакт тільки для top_k найкращих моделей за val_mae."""
    runs = {}
    for cfg in configs:
        print(f"\n=== {cfg['name']} (features={cfg['features']}, window={cfg['window']}) ===")
        runs[cfg["name"]] = run_experiment(cfg)

    table = pd.DataFrame([run["row"] for run in runs.values()])

    # Сортуємо результати за кращим val_mae
    sorted_table = table.sort_values("val_mae").reset_index(drop=True)
    top_runs = sorted_table.head(top_k)

    print(f"\n Зберігаємо артефакти для TOP-{top_k} моделей...")

    for _, top_row in top_runs.iterrows():
        run_name = top_row["name"]
        run_id = top_row["run_id"]
        exp_result = runs[run_name]

        # Дозаписуємо артефакт збереженої моделі у відповідний MLflow run
        with mlflow.start_run(run_id=run_id):
            mlflow.pytorch.log_model(
                exp_result["model"],
                artifact_path="model",
                serialization_format="pickle"
            )
            print(f" Збережено артефакт моделі '{run_name}' (val_mae: {top_row['val_mae']:.3f}) у run_id: {run_id}")

    return sorted_table, runs

EXPERIMENTS = [
    {"name": "base_w30", "features": "base", "window": 30, "epochs": 30, "patience": 5},
    {"name": "degr_w30_e100", "features": "degradation", "window": 30, "epochs": 100, "patience": 5},
    {"name": "degr_w30", "features": "degradation", "window": 30, "epochs": 60, "patience": 7},
    {"name": "degr_w50", "features": "degradation", "window": 50, "epochs": 60, "patience": 7},
    {"name": "degr_w80", "features": "degradation", "window": 80, "epochs": 60, "patience": 7},
]

In [20]:
results_df, runs = run_grid(EXPERIMENTS)


=== base_w30 (features=base, window=30) ===
base_w30 | ep 01 | Train Loss 11163.195 | Val MAE 74.958
base_w30 | ep 02 | Train Loss 8711.578 | Val MAE 62.945
base_w30 | ep 03 | Train Loss 6853.428 | Val MAE 52.556
base_w30 | ep 04 | Train Loss 5338.966 | Val MAE 43.737
base_w30 | ep 05 | Train Loss 4162.296 | Val MAE 36.752
base_w30 | ep 06 | Train Loss 3342.321 | Val MAE 32.144
base_w30 | ep 07 | Train Loss 2788.844 | Val MAE 28.963
base_w30 | ep 08 | Train Loss 2422.617 | Val MAE 27.171
base_w30 | ep 09 | Train Loss 2150.101 | Val MAE 24.999
base_w30 | ep 10 | Train Loss 1926.218 | Val MAE 24.125
base_w30 | ep 11 | Train Loss 1737.008 | Val MAE 22.277
base_w30 | ep 12 | Train Loss 1581.892 | Val MAE 21.847
base_w30 | ep 13 | Train Loss 1453.412 | Val MAE 21.521
base_w30 | ep 14 | Train Loss 1343.668 | Val MAE 20.832
base_w30 | ep 15 | Train Loss 1251.291 | Val MAE 20.178
base_w30 | ep 16 | Train Loss 1168.545 | Val MAE 19.829
base_w30 | ep 17 | Train Loss 1080.442 | Val MAE 19.786
ba

In [21]:
results_df, runs = run_grid_and_save_top_k(EXPERIMENTS, top_k=1)


=== base_w30 (features=base, window=30) ===
base_w30 | ep 01 | Train Loss 11163.195 | Val MAE 74.958
base_w30 | ep 02 | Train Loss 8711.578 | Val MAE 62.945
base_w30 | ep 03 | Train Loss 6853.428 | Val MAE 52.556
base_w30 | ep 04 | Train Loss 5338.966 | Val MAE 43.737
base_w30 | ep 05 | Train Loss 4162.296 | Val MAE 36.752
base_w30 | ep 06 | Train Loss 3342.321 | Val MAE 32.144
base_w30 | ep 07 | Train Loss 2788.844 | Val MAE 28.963
base_w30 | ep 08 | Train Loss 2422.617 | Val MAE 27.171
base_w30 | ep 09 | Train Loss 2150.101 | Val MAE 24.999
base_w30 | ep 10 | Train Loss 1926.218 | Val MAE 24.125
base_w30 | ep 11 | Train Loss 1737.008 | Val MAE 22.277
base_w30 | ep 12 | Train Loss 1581.892 | Val MAE 21.847
base_w30 | ep 13 | Train Loss 1453.412 | Val MAE 21.521
base_w30 | ep 14 | Train Loss 1343.668 | Val MAE 20.832
base_w30 | ep 15 | Train Loss 1251.291 | Val MAE 20.178
base_w30 | ep 16 | Train Loss 1168.545 | Val MAE 19.829
base_w30 | ep 17 | Train Loss 1080.442 | Val MAE 19.786
ba

2026/11/07 12:49:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/07 12:49:56 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/11/07 12:49:56 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.14.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/07 12:50:04 WARNING mlflow.utils.requirements_utils: Found torch version (2.14.1+cu130) contains a local version label

 Збережено артефакт моделі 'degr_w80' (val_mae: 7.785) у run_id: 260d2ed7487f442083142efbe7a5e2d1
🏃 View run degr_w80 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/260d2ed7487f442083142efbe7a5e2d1
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2


# Збереження

In [ ]:
import mlflow.pytorch
import torch
from google.colab import files


RUN_ID = ...

model_uri = f"runs:/{RUN_ID}/model"
loaded_model = mlflow.pytorch.load_model(model_uri)

torch.save(loaded_model.state_dict(), "best_lstm_degr_w80.pt")

files.download("best_lstm_degr_w80.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Результати

In [ ]:
results_df = results_df.sort_values("val_mae").reset_index(drop=True)
results_df.round(3)

In [ ]:
fig, axes = plt.subplots(len(runs), 2, figsize=(14, 4 * len(runs)), squeeze=False)
for row, (name, run) in enumerate(runs.items()):
    hist = run["history"]
    epochs = np.arange(1, len(hist["train_mse"]) + 1)
    best_epoch = run["row"]["best_epoch"]
    for col, (train_key, val_key, label) in enumerate([("train_mse", "val_mse", "MSE"),
                                                       ("train_mae", "val_mae", "MAE")]):
        ax = axes[row, col]
        ax.plot(epochs, hist[train_key], label=f"Train {label}")
        ax.plot(epochs, hist[val_key], label=f"Validation {label}")
        ax.axvline(best_epoch, color="red", linestyle="--", label=f"Best epoch: {best_epoch}")
        ax.set_title(f"{name}: {label}")
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_run = runs[results_df.iloc[0]["name"]]
actual, pred = best_run["actual"], best_run["pred"]

plt.figure(figsize=(7, 7))
plt.scatter(actual, pred, alpha=0.6)
lo, hi = min(actual.min(), pred.min()), max(actual.max(), pred.max())
plt.plot([lo, hi], [lo, hi], "r--", label="Perfect prediction")
plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")
plt.title(f"LSTM ({best_run['cfg']['name']}): Actual vs Predicted RUL")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"y_val: {actual.min():.0f}..{actual.max():.0f} | "
      f"передбачень > 150: {(pred > 150).sum()} з {len(pred)}")

### Збереження

In [ ]:
SAVE_DIR = DATA_DIR / "lstm"
ckpt_run = runs["degr_w80"]
ckpt_row = ckpt_run["row"]
features = DATASETS["degradation"]["features"]

if QUICK:
    print("QUICK-режим: чекпоінт не зберігаємо")
else:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state_dict": ckpt_run["state"],
            "window_size": ckpt_run["cfg"]["window"],
            "input_size": len(features),
            "hidden_size": HIDDEN_SIZE,
            "num_layers": 1,
            "best_epoch": int(ckpt_row["best_epoch"]),
            "val_mae": float(ckpt_row["val_mae"]),
            "val_rmse": float(ckpt_row["val_rmse"]),
            "high_rul_mae_150+": float(ckpt_row["high_rul_mae_150+"]),
            "features": list(features),
        },
        SAVE_DIR / "lstm_fd001_best.pt",
    )
    joblib.dump(DATASETS["degradation"]["scaler"], SAVE_DIR / "scaler_fd001.pkl")
    print("Збережено файли:", sorted(os.listdir(SAVE_DIR)))

## Висновки

Числа з оригінальних прогонів (validation, вибір за MAE):

| Конфігурація | Ознаки | Вікно | Best epoch | Val MAE | Val RMSE | R² |
|---|---|---|---|---|---|---|
| base_w30 | base, 25 | 30 | 29 | 18.62 | 26.78 | 0.789 |
| degr_w30_e100 | degradation + індикатор, 27 | 30 | 28 | 15.10 | 24.52 | 0.830 |
| degr_w30 | degradation + індикатор, 27 | 30 | 29 | 16.41 | 26.51 | — |
| degr_w50 | degradation + індикатор, 27 | 50 | 35 | 11.68 | 20.08 | — |
| degr_w80 | degradation + індикатор, 27 | 80 | 40 | 9.07 | 14.92 | — |

- Найкраще вікно — 80 (val MAE 9.07). При однакових параметрах навчання (60 епох, patience 7) MAE падає з 16.41 (вікно 30) до 11.68 (50) і 9.07 (80).
- Degradation-набір із індикатором дає менший MAE, ніж базові 25 ознак (15.10 проти 18.62). Прогони з різними параметрами навчання порівнюються обережно.
- MAE для двигунів з RUL >= 150 лишається близько 35 у всіх вікнах (35.3–35.5), тобто помилка на великих RUL від вікна не залежить.
- R² для вікон 30/50/80 в оригіналі не виводився (прочерк).
- Усі цифри вище — validation, за яким вибиралась епоха. Офіційний тест тут не оцінювали.